## Dark count rate calculation 

Author:  Areg Danagoulian
Date: 06.02.2025

This code determines the rate at which the dark counts in a silicon photomultiplier (SiPM) of the model AFBR-S4N44P044M simultaneously achieve a value above some threshold. This is necessary for determining the minimum sensitivity of the SiPM.  

Specifically, the problem is the following:  in our system the inner detector is operated in !coincidence with the external detector, as a way of suppressing proton tracks while detecting neutron tracks.  If a random neutron trigger in the *inner* detector achieves a false coincidence with the dark counts above the threshold in the *outer* detector, that neutron will not produce an event, amounting to a false negative. We want to quantify the value of the false negatives for a given threshold. 

In [25]:
from scipy.stats import poisson;
from scipy.stats import norm
import math

time=365.25*24*3600 #seconds in a year
dt  = 2e-9 #integration window, assuming a peak detector circuit, and a 2ns rise-decay time in the scintillator (e.g. BC-408)
detector_size = 10 # det size in mm
detector_number = 1000 # number of detectors in the system

dc_rate = 125e+3 #counts per mm2, AFBR-S4N44P044M
pitch = 40e-3 #size of a cell
quantum_efficiency = 0.63
fill_factor=3.72*3.62/6**2 #about 3.6m, for a total size of 6

In [26]:
rate = dc_rate * pitch**2 #the rate in a cell, per second
p    = rate*dt #the probability of a count within a dt, in a given cell
print(p)

4.0000000000000003e-07


In [27]:
p_detector = p * (detector_size/pitch)**2 #the expected counts in a 1x1 cm2 detector 
threshold = 5 # photon count threshold, based on the expected number of photoelectrons in a signal (50)
e_threshold = threshold*quantum_efficiency*fill_factor #the threshold in terms of photoelectrons, after QEff and fill factor are taken account
probability_above_threshold = 1 - poisson.cdf(e_threshold-1,p_detector) #probability, in a given dt window, of getting counts above e_threshold
#Note on the below:  actually, the only reason we care about this is because we are worried that a legitimate neutron hit in the inner detector
#will coincide with a dark count trigger in the outer.  So what matters is only the probability of scoring X dark counts simultaneously in a given
#dt window! That's it!
#print(f'Probability per second: {probability_above_threshold*(1/dt):.32f}') #the probability in one second, need to divide by dt.
print(f'Probability per array per second (FN probability): {probability_above_threshold:.4f}') #the probability in a given dt


Probability per array per second (FN probability): 0.0247


But this is not all.  The outer detector needs to be able to detect a trigger, defined by a production of *photons_expected* number of optical photons.  *photons_expected* is a mean expected photon count, not the actual count!  What is the probability that one will see less than that many optical photons, and as such the outer detector will fail to produce a veto, thus resulting in a false neutron count? 

In [28]:
#Now let's calculate the probability, assuming 50 expected optical photons , the chance of missing a hit in the outer detector
photons_expected=50
z = (threshold-photons_expected)/math.sqrt(photons_expected)
print(f'The False Negative rate in the outer detector is: {norm.cdf(z)}')
print(f'Expected number of misses, given 50e+3 hits per second per detector, is: {50e+3*norm.cdf(z)}')
print(f'Same, for the whole array, per second...this is the false positive neutron rate: {detector_number*50e+3*norm.cdf(z):.4f}')


The False Negative rate in the outer detector is: 9.830802207714438e-11
Expected number of misses, given 50e+3 hits per second per detector, is: 4.915401103857219e-06
Same, for the whole array, per second...this is the false positive neutron rate: 0.0049
